# Customer / Movie Review Sentiment Analysis Using NLP
**Datheon Machine Learning & Data Science Internship — Sentiment Classification Task**

Classify positive vs negative reviews with **TF-IDF**, **Multinomial Naive Bayes**, and **Logistic Regression**. Run all cells and save the notebook with actual outputs before submitting to GitHub. Results are **not precomputed**.

In [ ]:
%pip install -q pandas numpy matplotlib seaborn scikit-learn nltk

In [ ]:
import re, html
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

for resource in ['stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(resource, quiet=True)
STOPWORDS = set(stopwords.words('english')) - {'not', 'no', 'nor'}
LEMMATIZER = WordNetLemmatizer()
RANDOM_STATE = 42

## 1. Load official dataset
The notebook first checks for `sentiment_reviews.csv` locally, then downloads from the Datheon task link. If the URL is unavailable, manually download the dataset and put it in the same folder as this notebook.

In [ ]:
csv_path = Path('sentiment_reviews.csv')
url = 'https://www.datheon.in/datasets/sentiment_reviews.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
else:
    try:
        df = pd.read_csv(url)
        df.to_csv(csv_path, index=False)
    except Exception as exc:
        raise FileNotFoundError('Dataset could not be downloaded. Save the official CSV as sentiment_reviews.csv beside the notebook, then rerun.') from exc
print('Dataset shape:', df.shape)
print('Columns:', df.columns.tolist())
display(df.head())

## 2. Identify review text and sentiment labels
Common column names are supported. If your downloaded CSV has different names, update `text_col` and `label_col` manually.

In [ ]:
colmap = {str(c).lower().strip().replace(' ', '_'): c for c in df.columns}
text_col = next((colmap[c] for c in ['review', 'text', 'review_text', 'content', 'sentence', 'comment', 'feedback'] if c in colmap), None)
label_col = next((colmap[c] for c in ['sentiment', 'label', 'target', 'polarity', 'class'] if c in colmap), None)
if text_col is None or label_col is None:
    raise ValueError(f'Cannot identify review and sentiment columns: {df.columns.tolist()}. Set text_col and label_col explicitly.')
print('Text column:', text_col, '| Label column:', label_col)
data = df[[text_col, label_col]].copy().dropna()
data.columns = ['review', 'sentiment']
data['review'] = data['review'].astype(str)
labels = data['sentiment'].astype(str).str.lower().str.strip()
label_mapping = {'positive':1, 'negative':0, 'pos':1, 'neg':0, '1':1, '0':0, '1.0':1, '0.0':0}
data['target'] = labels.map(label_mapping)
if data['target'].isna().any():
    raise ValueError(f'Unexpected sentiment labels: {sorted(labels.unique().tolist())}. Update label_mapping to match your dataset.')
data = data[data['review'].str.strip().ne('')].drop_duplicates(subset=['review', 'target'])
if data['target'].nunique() != 2:
    raise ValueError('Both positive and negative labels are required.')
print('Class counts:')
display(data['target'].map({0:'Negative',1:'Positive'}).value_counts())

## 3. Clean, tokenize, remove stopwords, and lemmatize

In [ ]:
def clean_text(text):
    text = html.unescape(str(text)).lower()
    text = re.sub(r'<[^>]*>', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = re.findall(r'[a-z]+', text)
    tokens = [LEMMATIZER.lemmatize(token) for token in tokens if token not in STOPWORDS]
    return ' '.join(tokens)

print('Example original:', data['review'].iloc[0])
print('Example cleaned:', clean_text(data['review'].iloc[0]))
# Remove texts that become empty after preprocessing.
data['cleaned'] = data['review'].apply(clean_text)
data = data[data['cleaned'].str.len().gt(0)].copy()
print('Rows after cleaning:', len(data))

## 4. Stratified 80/20 split and model training
TF-IDF is fitted **only on training data** inside the pipeline to avoid data leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data['cleaned'], data['target'].astype(int),
    test_size=0.2, random_state=RANDOM_STATE, stratify=data['target']
)
models = {
    'Multinomial Naive Bayes': MultinomialNB(alpha=1.0),
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
}
trained = {}
results = []
for name, classifier in models.items():
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
        ('classifier', classifier)
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    trained[name] = pipe
    results.append({'Model': name, 'Test accuracy': accuracy_score(y_test, preds)})
results_df = pd.DataFrame(results).sort_values('Test accuracy', ascending=False)
display(results_df)

## 5. Classification accuracy benchmarks and reports

In [ ]:
for name, pipe in trained.items():
    print('\nMODEL:', name)
    print(classification_report(y_test, pipe.predict(X_test), target_names=['Negative','Positive'], zero_division=0))
# Optional stratified CV on training partition only
if y_train.value_counts().min() >= 5:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    for name, classifier in models.items():
        pipe = Pipeline([('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))), ('classifier', classifier)])
        scores = cross_val_score(pipe, X_train, y_train, scoring='accuracy', cv=cv)
        print(f'{name} 5-fold training CV accuracy: {scores.mean():.3f} +/- {scores.std():.3f}')

## 6. Confusion matrices and accuracy comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, pipe) in zip(axes, trained.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, pipe.predict(X_test),
        display_labels=['Negative','Positive'], cmap='Blues', colorbar=False, ax=ax)
    ax.set_title(name)
plt.tight_layout()
plt.savefig('sentiment_confusion_matrices.png', dpi=180)
plt.show()

ax = sns.barplot(data=results_df, x='Model', y='Test accuracy')
ax.set_ylim(0,1)
ax.set_title('Test accuracy comparison')
plt.xticks(rotation=12)
plt.tight_layout()
plt.savefig('sentiment_accuracy_comparison.png', dpi=180)
plt.show()

## 7. Live `predict_sentiment()` test helper
This function accepts any custom English review and returns its predicted sentiment and probability.

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model = trained[best_name]

def predict_sentiment(review_text):
    cleaned = clean_text(review_text)
    if not cleaned:
        return {'error': 'Please enter a review containing meaningful words.'}
    label = int(best_model.predict([cleaned])[0])
    probabilities = best_model.predict_proba([cleaned])[0]
    class_position = list(best_model.classes_).index(label)
    return {'review': review_text, 'sentiment': 'Positive' if label == 1 else 'Negative',
            'confidence': round(float(probabilities[class_position]), 4), 'model': best_name}

for example in ['This movie was amazing and I loved it!',
                'Terrible product, very disappointing and not worth buying.',
                'The experience was not good at all.']:
    print(predict_sentiment(example))

# Try your own sentence:
predict_sentiment('I really enjoyed this product!')

## 8. Conclusion
The notebook compares Naive Bayes and Logistic Regression on a held-out test set. After executing all cells, write your **actual measured** accuracy values in your GitHub README. Accuracy depends on the dataset and is not guaranteed. The test helper is intended for demonstration, not production deployment.